In [37]:
import pandas as pd

station = pd.read_parquet("../../Data/sort_data/2024_data.parquet")
station2025 = pd.read_parquet("../../Data/sort_data/2025_data.parquet")

In [38]:
# 특정 정류소가 시작이거나 종료인 데이터만 반환
def filter_station_rows(df, keyword):
    mask = (
        df['시작_대여소_ID'].astype(str).str.contains(keyword, na=False) |
        df['종료_대여소_ID'].astype(str).str.contains(keyword, na=False)
    )
    
    filtered_df = df[mask].copy()
    
    return filtered_df

In [39]:
# 종료가 해당 정류소인 경우 도착시간으로 변경
import numpy as np

def update_end(df, station_id):
    mask = df["종료_대여소_ID"] == station_id

    # Timedelta -> 시간(실수)
    hours = df.loc[mask, "전체_이용_분"].dt.total_seconds() / 3600.0

    # 버림 처리
    raw = np.floor(df.loc[mask, "시간대"] - hours).astype(int)

    # 음수 시간 처리: 기준_날짜 하루(또는 여러 일) 줄이고 시간대 보정
    day_shift = np.floor_divide(raw, 24)  # 음수면 -1, -2 ...
    adj_hour = raw - day_shift * 24       # 0~23로 보정

    # 날짜 업데이트
    df.loc[mask, "기준_날짜"] = (
        pd.to_datetime(df.loc[mask, "기준_날짜"]) + pd.to_timedelta(day_shift, unit="D")
    )

    # 시간대 업데이트
    df.loc[mask, "시간대"] = adj_hour

    # 집계 기준 변경
    df.loc[mask, "집계_기준"] = "도착시간"

    # 기준_날짜 -> 시간대 정렬
    df = df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)
    return df


In [40]:
# 모든 경우의 수 행 전부 생성, 출발 건수 마이너스 처리
def build_hourly_df(df):
    df = df.copy()

    # 불필요 컬럼 드랍
    df = df.drop(columns=["전체_이용_분", "전체_이용_거리"], errors="ignore")

    # 기준_날짜를 문자열(YYYY-MM-DD)로 통일
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"]).dt.strftime("%Y-%m-%d")

    # 시간 범위 생성
    hours = pd.date_range("2024-01-01 00:00", "2025-12-31 23:00", freq="h")

    base = pd.DataFrame({"_dt": hours})
    base = pd.concat(
        [
            base.assign(집계_기준="출발시간"),
            base.assign(집계_기준="도착시간"),
        ],
        ignore_index=True,
    )
    base["기준_날짜"] = base["_dt"].dt.strftime("%Y-%m-%d")
    base["시간대"] = base["_dt"].dt.hour
    base = base.drop(columns=["_dt"])

    # 전체_건수 합산
    counts = (
        df.groupby(["기준_날짜", "시간대", "집계_기준"], as_index=False)["전체_건수"]
        .sum()
    )
    base = base.merge(counts, on=["기준_날짜", "시간대", "집계_기준"], how="left")

    # 출발시간은 합계에 마이너스
    mask_arr = base["집계_기준"] == "출발시간"
    base.loc[mask_arr, "전체_건수"] = -base.loc[mask_arr, "전체_건수"]

    # 없으면 0으로
    base["전체_건수"] = base["전체_건수"].fillna(0)

    # 나머지 컬럼은 출발시간 기준으로만 채우기
    other_cols = [
        c for c in df.columns
        if c not in ["기준_날짜", "시간대", "집계_기준", "전체_건수"]
    ]
    if other_cols:
        dep = df[df["집계_기준"] == "출발시간"]
        meta = dep.groupby(["기준_날짜", "시간대"], as_index=False)[other_cols].first()
        base = base.merge(meta, on=["기준_날짜", "시간대"], how="left")

        # 도착시간 행은 기타 컬럼을 NaN으로
        base.loc[mask_arr, other_cols] = pd.NA

    # 기준_날짜 → 시간대 정렬
    base = base.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)

    return base


In [41]:
# 날씨 빈 부분 채우기, 불필요 컬럼 제거
import requests

def add_open_meteo_weather(df, lat=37.60287, lon=126.93023, tz="Asia/Seoul"):
    df = df.copy()

    # 1) 컬럼 드랍
    df = df.drop(columns=["시작_대여소_ID", "종료_대여소_ID", "불쾌지수", "온도", "습도", "강수량", "적설량"], errors="ignore")

    # 2) 날짜 범위 결정
    dates = pd.to_datetime(df["기준_날짜"])
    start_date = dates.min().date().isoformat()
    end_date = dates.max().date().isoformat()

    # 3) Open-Meteo Historical Weather API 호출
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall",
        "timezone": tz,
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    # 4) 시간별 날씨 df 만들기
    hourly = data["hourly"]
    weather = pd.DataFrame({
        "time": pd.to_datetime(hourly["time"]),
        "온도": hourly["temperature_2m"],
        "습도": hourly["relative_humidity_2m"],
        "강수량": hourly["precipitation"],
        "적설량": hourly["snowfall"],
    })
    weather["기준_날짜"] = weather["time"].dt.strftime("%Y-%m-%d")
    weather["시간대"] = weather["time"].dt.hour
    weather = weather.drop(columns=["time"])

    # 5) 원본 df에 합치기
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"]).dt.strftime("%Y-%m-%d")
    df = df.merge(weather, on=["기준_날짜", "시간대"], how="left")

    return df


In [42]:
# 출퇴근시간, 요일, 주말 컬럼 추가
def add_time_flags(df, commute_hours=None):
    df = df.copy()

    # 출퇴근 시간대 기본값
    if commute_hours is None:
        commute_hours = [7, 8, 9, 17, 18, 19]

    # 날짜 파싱
    date = pd.to_datetime(df["기준_날짜"])

    # 요일 컬럼 (월~일)
    weekday_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
    df["요일"] = date.dt.weekday.map(weekday_map)

    # 출퇴근시간 컬럼
    df["출퇴근시간"] = df["시간대"].isin(commute_hours).astype(int)

    # 2024년 공휴일 (한국)
    holidays_2024 = pd.to_datetime([
        "2024-01-01",
        "2024-02-09", "2024-02-10", "2024-02-11", "2024-02-12",
        "2024-03-01",
        "2024-04-10",
        "2024-05-05", "2024-05-06",
        "2024-05-15",
        "2024-06-06",
        "2024-08-15",
        "2024-09-16", "2024-09-17", "2024-09-18",
        "2024-10-01",
        "2024-10-03",
        "2024-10-09",
        "2024-12-25",
    ])

    # 주말 컬럼 (토/일 또는 2024 공휴일)
    is_weekend = date.dt.weekday >= 5
    is_holiday_2024 = date.isin(holidays_2024)
    df["주말"] = (is_weekend | is_holiday_2024).astype(int)

    return df


In [43]:
import numpy as np
import pandas as pd


def add_time_features(
    df: pd.DataFrame,
    group_cols=None,
    date_col: str = "기준_날짜",
    hour_col: str = "시간대",
    target_col: str = "전체_건수",
) -> pd.DataFrame:
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    sort_cols = [date_col, hour_col]
    if group_cols:
        sort_cols = list(group_cols) + sort_cols
    df = df.sort_values(sort_cols).reset_index(drop=True)

    df["dow"] = df[date_col].dt.dayofweek
    df["day_type"] = (df["dow"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df[hour_col] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df[hour_col] / 24.0)

    if group_cols:
        g = df.groupby(group_cols, sort=False)
        df["lag1"] = g[target_col].shift(1)
        df["lag3"] = g[target_col].shift(3)
        df["lag24"] = g[target_col].shift(24)
        df["lag48"] = g[target_col].shift(48)
        df["lag168"] = g[target_col].shift(168)
        df["rolling3"] = (
            g[target_col].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
        )
        df["rolling24"] = (
            g[target_col].shift(1).rolling(24).mean().reset_index(level=0, drop=True)
        )
    else:
        df["lag1"] = df[target_col].shift(1)
        df["lag3"] = df[target_col].shift(3)
        df["lag24"] = df[target_col].shift(24)
        df["lag48"] = df[target_col].shift(48)
        df["lag168"] = df[target_col].shift(168)
        df["rolling3"] = df[target_col].shift(1).rolling(3).mean()
        df["rolling24"] = df[target_col].shift(1).rolling(24).mean()

    df["rush_hour"] = (
        ((df[hour_col] >= 7) & (df[hour_col] <= 9))
        | ((df[hour_col] >= 17) & (df[hour_col] <= 19))
    ).astype(int)

    return df


def add_discomfort_index(
    df: pd.DataFrame,
    temp_col: str = "온도",
    humidity_col: str = "습도",
    out_col: str = "불쾌지수",
) -> pd.DataFrame:
    df = df.copy()
    # Common THI/Discomfort Index used in Korea:
    # DI = 0.81*T + 0.01*RH*(0.99*T - 14.3) + 46.3
    df[out_col] = (
        0.81 * df[temp_col]
        + 0.01 * df[humidity_col] * (0.99 * df[temp_col] - 14.3)
        + 46.3
    )
    return df


def make_features(
    df: pd.DataFrame,
    group_cols=None,
    date_col: str = "기준_날짜",
    hour_col: str = "시간대",
    temp_col: str = "온도",
    humidity_col: str = "습도",
    rain_col: str = "강수량",
    snow_col: str = "적설량",
    target_col: str = "전체_건수",
) -> pd.DataFrame:
    required = [date_col, hour_col, temp_col, humidity_col, rain_col, snow_col, target_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    df_feat = add_time_features(
        df,
        group_cols=group_cols,
        date_col=date_col,
        hour_col=hour_col,
        target_col=target_col,
    )
    df_feat = add_discomfort_index(
        df_feat, temp_col=temp_col, humidity_col=humidity_col, out_col="불쾌지수"
    )
    return df_feat


In [44]:
st_2247 = filter_station_rows(station, "ST-2247")
st_2252 = filter_station_rows(station, "ST-2252")
st_2425 = filter_station_rows(station, "ST-2425")

st_2247_test = filter_station_rows(station2025, "ST-2247")
st_2252_test = filter_station_rows(station2025, "ST-2252")
st_2425_test = filter_station_rows(station2025, "ST-2425")

In [45]:
st_2247 = update_end(st_2247, "ST-2247")
st_2252 = update_end(st_2252, "ST-2252")
st_2425 = update_end(st_2425, "ST-2425")

st_2247_test = update_end(st_2247_test, "ST-2247")
st_2252_test = update_end(st_2252_test, "ST-2252")
st_2425_test = update_end(st_2425_test, "ST-2425")

In [46]:
st_2247 = build_hourly_df(st_2247)
st_2252 = build_hourly_df(st_2252)
st_2425 = build_hourly_df(st_2425)

st_2247_test = build_hourly_df(st_2247_test)
st_2252_test = build_hourly_df(st_2252_test)
st_2425_test = build_hourly_df(st_2425_test)

In [ ]:
st_2247 = add_open_meteo_weather(st_2247)
st_2252 = add_open_meteo_weather(st_2252)
st_2425 = add_open_meteo_weather(st_2425)

st_2247_test = add_open_meteo_weather(st_2247_test)
st_2252_test = add_open_meteo_weather(st_2252_test)
st_2425_test = add_open_meteo_weather(st_2425_test)

In [48]:
st_2247 = add_time_flags(st_2247)
st_2252 = add_time_flags(st_2252)
st_2425 = add_time_flags(st_2425)

st_2247_test = add_time_flags(st_2247_test)
st_2252_test = add_time_flags(st_2252_test)
st_2425_test = add_time_flags(st_2425_test)

In [50]:
st_2247

,집계_기준_x,기준_날짜,시간대,전체_건수_x,시작_대여소_ID,종료_대여소_ID,온도_x,습도_x,불쾌지수,강수량_x,적설량_x,집계_기준_y,전체_건수_y,온도_y,습도_y,강수량_y,적설량_y,요일,출퇴근시간,주말
0,출발시간,2024-01-01,0,-2.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,출발시간,-2.0,-1.9,91,0.0,0.0,월,0,1
1,출발시간,2024-01-01,0,-2.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,도착시간,0.0,-1.9,91,0.0,0.0,월,0,1
2,도착시간,2024-01-01,0,0.0,ST-2247,ST-463,-2.7,92.0,28.49784,0.0,0.0,출발시간,-2.0,-1.9,91,0.0,0.0,월,0,1
3,도착시간,2024-01-01,0,0.0,ST-2247,ST-463,-2.7,92.0,28.49784,0.0,0.0,도착시간,0.0,-1.9,91,0.0,0.0,월,0,1
4,출발시간,2024-01-01,1,0.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,출발시간,0.0,-2.8,91,0.0,0.0,월,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70171,도착시간,2025-12-31,22,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,도착시간,0.0,-8.4,52,0.0,0.0,수,0,0
70172,출발시간,2025-12-31,23,0.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,출발시간,0.0,-8.8,52,0.0,0.0,수,0,0
70173,출발시간,2025-12-31,23,0.0,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,도착시간,0.0,-8.8,52,0.0,0.0,수,0,0
70174,도착시간,2025-12-31,23,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,출발시간,0.0,-8.8,52,0.0,0.0,수,0,0


In [49]:
st_2247 = make_features(st_2247)
st_2252 = make_features(st_2252)
st_2425 = make_features(st_2425)

st_2247_test = make_features(st_2247_test)
st_2252_test = make_features(st_2252_test)
st_2425_test = make_features(st_2425_test)

KeyError: "Missing columns: ['온도', '습도', '강수량', '적설량', '전체_건수']"

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd


def train_model_lgbm(
    df,
    df_test,
    time_col=("기준_날짜", "시간대"),
    feature_cols=("시간대", "day_type", "온도", "습도", "불쾌지수",
                  "강수량", "적설량", "lag1", "lag3", "lag24",
                  "hour_sin", "hour_cos", "rolling3", "rolling24",
                  "lag48", "lag168", "dow", "rush_hour"),
    target_col="전체_건수",
    random_state=42,
    n_splits=5,
):
    # ✅ 정렬
    df = df.copy()
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])
    df = df.sort_values(list(time_col)).reset_index(drop=True)

    df_test = df_test.copy()
    df_test["기준_날짜"] = pd.to_datetime(df_test["기준_날짜"])
    df_test = df_test.sort_values(list(time_col)).reset_index(drop=True)

    # ✅ NaN 제거
    df = df.dropna()
    df_test = df_test.dropna()

    X = df[list(feature_cols)]
    y = df[target_col]

    X_test = df_test[list(feature_cols)]
    y_test = df_test[target_col]

    # ✅ 모델
    model = LGBMRegressor(
        random_state=random_state,
        n_jobs=-1
    )

    param_grid = {
        "n_estimators": [300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [-1, 10],
        "num_leaves": [31, 63],
    }

    tscv = TimeSeriesSplit(n_splits=n_splits)

    search = GridSearchCV(
        model,
        param_grid,
        cv=tscv,
        scoring="r2",
        n_jobs=-1
    )

    search.fit(X, y)

    best_model = search.best_estimator_

    # -----------------------------
    # ✅ Train score
    pred_train = best_model.predict(X)
    train_score = {
        "r2": r2_score(y, pred_train),
        "mae": mean_absolute_error(y, pred_train),
        "rmse": np.sqrt(mean_squared_error(y, pred_train))
    }

    # -----------------------------
    # ✅ CV score (validation 역할)
    cv_score = search.best_score_

    # -----------------------------
    # ✅ Test score
    pred_test = best_model.predict(X_test)
    test_score = {
        "r2": r2_score(y_test, pred_test),
        "mae": mean_absolute_error(y_test, pred_test),
        "rmse": np.sqrt(mean_squared_error(y_test, pred_test))
    }

    scores = {
        "train": train_score,
        "cv_valid_r2_mean": cv_score,
        "test": test_score,
        "best_params": search.best_params_
    }

    return best_model, scores

In [ ]:
st_2247_test

In [ ]:
best_model, scores = train_model_lgbm(
    df=st_2247,
    df_test=st_2247_test,
    n_splits=3,
)

print(scores)

In [ ]:
# Feature importance (RandomForest)
import pandas as pd

feature_cols=("시간대", "day_type", "온도", "습도", "불쾌지수",
    "강수량", "적설량", "lag1", "lag3", "lag24",
    "hour_sin", "hour_cos", "rolling3", "rolling24",
    "lag48", "lag168", "dow", "rush_hour")

fi = pd.Series(
    best_model.feature_importances_,
    index=list(feature_cols)
).sort_values(ascending=False)

fi


In [ ]:
# Feature importance bar chart
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

fi = pd.Series(
    best_model.named_steps["model"].feature_importances_,
    index=list(feature_cols),
).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
fi.plot(kind="bar")
plt.title("Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Daily trend by month
import matplotlib.pyplot as plt

df_day = st_2247.copy()
df_day["기준_날짜"] = pd.to_datetime(df_day["기준_날짜"])
daily = (
    df_day
    .set_index("기준_날짜")["전체_건수"]
    .resample("D")
    .sum()
)

plt.figure(figsize=(10, 5))
for month, series in daily.groupby(daily.index.to_period("M")):
    plt.plot(series.index.day, series.values, label=str(month))

plt.title("Daily Totals by Month")
plt.xlabel("Day of Month")
plt.ylabel("Total Count")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()
